In [74]:
import pandas as pd

In [86]:
fundamentals=pd.read_csv("/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Merged files/csv/apple/merged_apple_financials_clean_09_25_complete.csv")
prices=pd.read_csv("/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Prices/csv/apple/apple_price.csv")
gdp=pd.read_csv("/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Macroeconomic/csv files/macro_2010_2025.csv")

In [87]:
fundamentals

,period_end,form,quarter,fq,fy,revenue,net_income,total_assets,total_liabilities,shareholders_equity
0,2009-12-26,10-Q,Q1 2010,1,2010,15683000000,3378000000,53926000000,18158000000,35768000000
1,2010-03-27,10-Q,Q2 2010,2,2010,13499000000,3074000000,57057000000,17709000000,39348000000
2,2010-06-26,10-Q,Q3 2010,3,2010,15700000000,3253000000,64725000000,21614000000,43111000000
3,2010-09-25,10-Q,Q4 2010,4,2010,20343000000,4308000000,75183000000,27392000000,47791000000
4,2010-12-25,10-Q,Q1 2011,1,2011,26741000000,6004000000,86742000000,32076000000,54666000000
...,...,...,...,...,...,...,...,...,...,...
58,2024-06-29,10-Q,Q3 2024,3,2024,85777000000,21448000000,331612000000,264904000000,66708000000
59,2024-09-28,10-Q,Q4 2024,4,2024,94930000000,14736000000,364980000000,308030000000,56950000000
60,2024-12-28,10-Q,Q1 2025,1,2025,124300000000,36330000000,344085000000,277327000000,66758000000
61,2025-03-29,10-Q,Q2 2025,2,2025,95359000000,24780000000,331233000000,264437000000,66796000000


In [89]:
#--FUNDAMENTALS DF--

#Convert period_end to datetime
fundamentals["period_end"] = pd.to_datetime(fundamentals["period_end"])


#Drop unecessary columns for fundamentals df - update if needed
fundamentals=fundamentals.drop(columns=["form","fq", "fy"])

#Create ratio assets/liabilities
fundamentals["ratio assets/libailities"] = fundamentals["total_assets"] / fundamentals["total_liabilities"]

#Drop assets and liabilities columns
fundamentals = fundamentals.drop(columns=["total_assets", "total_liabilities"])

#Change name of quarter to fiscal_quarter
fundamentals = fundamentals.rename(columns={"quarter": "fiscal_quarter"})

#Create column for calendar_quarter
# Calendar quarter pieces
p = fundamentals["period_end"].dt.to_period("Q")        # calendar quarters
fundamentals["cal_year"] = p.dt.year.astype("Int64")    # 2010, 2011, ...
fundamentals["cal_q"]    = p.dt.quarter                 # 1..4

# Final label: "Q{cal_q} {cal_year}"
fundamentals["calendar_quarter"] = "Q" + fundamentals["cal_q"].astype(str) + " " + fundamentals["cal_year"].astype(str)

In [90]:
fundamentals

,period_end,fiscal_quarter,revenue,net_income,shareholders_equity,ratio assets/libailities,cal_year,cal_q,calendar_quarter
0,2009-12-26,Q1 2010,15683000000,3378000000,35768000000,2.969820,2009,4,Q4 2009
1,2010-03-27,Q2 2010,13499000000,3074000000,39348000000,3.221921,2010,1,Q1 2010
2,2010-06-26,Q3 2010,15700000000,3253000000,43111000000,2.994587,2010,2,Q2 2010
3,2010-09-25,Q4 2010,20343000000,4308000000,47791000000,2.744706,2010,3,Q3 2010
4,2010-12-25,Q1 2011,26741000000,6004000000,54666000000,2.704265,2010,4,Q4 2010
...,...,...,...,...,...,...,...,...,...
58,2024-06-29,Q3 2024,85777000000,21448000000,66708000000,1.251820,2024,2,Q2 2024
59,2024-09-28,Q4 2024,94930000000,14736000000,56950000000,1.184885,2024,3,Q3 2024
60,2024-12-28,Q1 2025,124300000000,36330000000,66758000000,1.240719,2024,4,Q4 2024
61,2025-03-29,Q2 2025,95359000000,24780000000,66796000000,1.252597,2025,1,Q1 2025


In [91]:
#--PRICES DF--

#Keep only prices for the last day of each quarter

# ensure it's datetime
prices["Date"] = pd.to_datetime(prices["Date"], utc=True).dt.tz_convert(None)

# keep only rows from 2010 onwards
prices = prices[prices["Date"].dt.year >= 2009]

#only keep dates 03-31, 06-30, 09-30, 12-31
prices = prices[prices["Date"].dt.is_month_end & prices["Date"].dt.month.isin([3, 6, 9, 12])]
prices = prices.sort_values(by="Date", ascending=True)
prices = prices.reset_index(drop=True)


In [92]:
prices

,Unnamed: 0,Date,Price,calendar_quarter
0,43,2009-03-31,3.155715,Q1 2009
1,46,2009-06-30,4.275764,Q2 2009
2,49,2009-09-30,5.564229,Q3 2009
3,52,2009-12-31,6.326138,Q4 2009
4,55,2010-03-31,7.054726,Q1 2010
...,...,...,...,...
61,226,2024-06-30,209.401932,Q2 2024
62,229,2024-09-30,231.920639,Q3 2024
63,232,2024-12-31,249.534180,Q4 2024
64,235,2025-03-31,221.587616,Q1 2025


In [93]:
# -- GDP DF --

# 1) Ensure gdp["Date"] is datetime (not prices)
gdp["Date"] = pd.to_datetime(gdp["Date"], errors="coerce", utc=True).dt.tz_convert(None)
# If your column is named "DATE" (common from FRED), uncomment:
# gdp = gdp.rename(columns={"DATE": "Date"})

# 2) Calendar quarter label
p = gdp["Date"].dt.to_period("Q")                 # calendar quarters
gdp["cal_year"] = p.dt.year.astype("Int64")       # 2010, 2011, ...
gdp["cal_q"]    = p.dt.quarter                    # 1..4
gdp["calendar_quarter"] = "Q" + gdp["cal_q"].astype(str) + " " + gdp["cal_year"].astype(str)

# 3) Drop Date if you want (no trailing dot)
gdp = gdp.drop(columns=["Date"])




In [94]:
gdp

,quarter,gdp_growth,interest_rate,cal_year,cal_q,calendar_quarter
0,Q3 2025,1.99,4.330000,2025,2,Q2 2025
1,Q2 2025,1.99,4.330000,2025,1,Q1 2025
2,Q1 2025,2.53,4.650000,2024,4,Q4 2024
3,Q4 2024,2.72,5.263333,2024,3,Q3 2024
4,Q3 2024,3.04,5.330000,2024,2,Q2 2024
...,...,...,...,...,...,...
58,Q1 2011,2.78,0.186667,2010,4,Q4 2010
59,Q4 2010,3.34,0.186667,2010,3,Q3 2010
60,Q3 2010,2.91,0.193333,2010,2,Q2 2010
61,Q2 2010,1.75,0.133333,2010,1,Q1 2010


In [95]:
#merge funadmentals and prices dataframes
merged = pd.merge(fundamentals, prices, on="calendar_quarter", how="left")

#merge gdo

merged = merged.merge(gdp, on="calendar_quarter", how="left")

#Reorder columns
merged = merged[["calendar_quarter","period_end","Price","revenue", "net_income", "ratio assets/libailities", "shareholders_equity", "gdp_growth", "interest_rate"]]

merged.to_csv("merged_microsoft_funadmentals_price_macro.csv", index=False)

In [96]:
cols=["revenue","net_income","shareholders_equity"]
growth_rate= {}
for col in cols:
    growth_rate[col] = merged[col].pct_change().mean(skipna=True)

growth_rate["gdp_growth"] = merged["gdp_growth"].mean()
growth_rate["interest"] = merged["interest_rate"].mean()
growth_rate

{'revenue': np.float64(0.07419601480296174),
 'net_income': np.float64(0.11209410685874319),
 'shareholders_equity': np.float64(0.013783186805274492),
 'gdp_growth': np.float64(2.3834920634920636),
 'interest': np.float64(1.3291005291005291)}

In [97]:
merged

,calendar_quarter,period_end,Price,revenue,net_income,ratio assets/libailities,shareholders_equity,gdp_growth,interest_rate
0,Q4 2009,2009-12-26,6.326138,15683000000,3378000000,2.969820,35768000000,0.11,0.120000
1,Q1 2010,2010-03-27,7.054726,13499000000,3074000000,3.221921,39348000000,1.75,0.133333
2,Q2 2010,2010-06-26,7.550960,15700000000,3253000000,2.994587,43111000000,2.91,0.193333
3,Q3 2010,2010-09-25,8.518208,20343000000,4308000000,2.744706,47791000000,3.34,0.186667
4,Q4 2010,2010-12-25,9.683289,26741000000,6004000000,2.704265,54666000000,2.78,0.186667
...,...,...,...,...,...,...,...,...,...
58,Q2 2024,2024-06-29,209.401932,85777000000,21448000000,1.251820,66708000000,3.04,5.330000
59,Q3 2024,2024-09-28,231.920639,94930000000,14736000000,1.184885,56950000000,2.72,5.263333
60,Q4 2024,2024-12-28,249.534180,124300000000,36330000000,1.240719,66758000000,2.53,4.650000
61,Q1 2025,2025-03-29,221.587616,95359000000,24780000000,1.252597,66796000000,1.99,4.330000
